In [1]:
%pip install --no-cache-dir pdfplumber

     ---------------------------------------- 0.0/43.7 kB ? eta -:--:--
     --------- ------------------------------ 10.2/43.7 kB ? eta -:--:--
     ----------------- -------------------- 20.5/43.7 kB 162.5 kB/s eta 0:00:01
     -------------------------- ----------- 30.7/43.7 kB 187.9 kB/s eta 0:00:01
     -------------------------------------- 43.7/43.7 kB 237.0 kB/s eta 0:00:00
     ---------------------------------------- 0.0/63.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/63.5 kB ? eta -:--:--
     ------------------- -------------------- 30.7/63.5 kB ? eta -:--:--
     ---------------------------------------- 63.5/63.5 kB 1.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/47.3 kB ? eta -:--:--
     -------- ------------------------------- 10.2/47.3 kB ? eta -:--:--
     ------------------------ ------------- 30.7/47.3 kB 435.7 kB/s eta 0:00:01
     -------------------------------------- 47.3/47.3 kB 475.0 kB/s eta 0:00:00
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import pdfplumber

In [3]:
PDF_PATH = "data/constitution_eng.pdf"
FIRST_PAGE = 33
LAST_PAGE = 283

Remove Footnotes

In [4]:
def crop_above_footnotes(page):
    # pdfplumer has different attributes for lines and rects, so we need to filter out the lines that are actually separators
    separators = [l for l in page.lines if abs(l["top"] - l["bottom"]) < 1 and (l["x1"] - l["x0"]) > 100] # a line could be horizontal or vertical, this line checks that
    if not separators:
        return page
    cut_y = min(l["top"] for l in separators)
    return page.crop((0, 0, page.width, cut_y))


In [ ]:
def drop_small_chars(page, min_size=7):
    return page.filter(lambda obj: obj.get("object_type") != "char" or obj["size"] >= min_size)

In [6]:
HEADER_RE = re.compile(r"^\d*\s*THE CONSTITUTION OF INDIA$"   # running title
                       r"|^\(Part [^)]*\)$"                    # (Part III.—...)
                       r"|^\d+$")                              # bare page number

def clean_page(page):
    page = crop_above_footnotes(page)
    page = drop_small_chars(page)
    text = page.extract_text() or ""
    return "\n".join(ln for ln in text.splitlines()
                     if not HEADER_RE.match(ln.strip()))

In [7]:
with pdfplumber.open(PDF_PATH) as pdf:
    page_texts = [clean_page(pdf.pages[i]) for i in range(FIRST_PAGE - 1, LAST_PAGE)]

articles_text = "\n".join(page_texts)
print(len(articles_text))

492608


Saving/Reading the articles

In [8]:
import os

CACHE = "data/articles_text.txt"

if os.path.exists(CACHE): # agar already exust karta hai, then read
    with open(CACHE, encoding="utf-8") as f:
        articles_text = f.read()
else:
    with pdfplumber.open(PDF_PATH) as pdf: # for the first time, extract and cache
        articles_text = "\n".join(clean_page(pdf.pages[i]) for i in range(FIRST_PAGE - 1, LAST_PAGE))
    with open(CACHE, "w", encoding="utf-8") as f:
        f.write(articles_text)

Some preprocessing

The idea is to separate the title amd the body

In [9]:
ARTICLE_RE = re.compile(
    r"^\[?(\d{1,3}[A-Z]{0,3})\.\s+"        # "19. " or "[21A. " at the start of a line
    r"([^—⎯\n]+(?:\n[^—⎯\n]+){0,2}?)"      # title: up to 3 lines, no dash inside
    r"\s*[—⎯]",                             # the title ends at an em-dash
    re.M,
)

In [10]:
matches = list(ARTICLE_RE.finditer(articles_text))
matches


[<re.Match object; span=(35, 71), match='1. Name and territory of the Union.—'>,
 <re.Match object; span=(406, 451), match='2. Admission or establishment of new States.—'>,
 <re.Match object; span=(567, 615), match='[2A. [Sikkim to be associated with the Union.].—'>,
 <re.Match object; span=(705, 797), match='3. Formation of new States and alteration of area>,
 <re.Match object; span=(2026, 2185), match='4. Laws made under articles 2 and 3 to provide fo>,
 <re.Match object; span=(2786, 2842), match='5. Citizenship at the commencement of the Constit>,
 <re.Match object; span=(3225, 3311), match='6. Rights of citizenship of certain persons who h>,
 <re.Match object; span=(4429, 4487), match='7. Rights of citizenship of certain migrants to P>,
 <re.Match object; span=(5147, 5232), match='8. Rights of citizenship of certain persons of In>,
 <re.Match object; span=(5960, 6044), match='9. Persons voluntarily acquiring citizenship of a>,
 <re.Match object; span=(6250, 6296), match='10. Contin

In [16]:
matches[0], matches[0].start(), matches[0].group(1), matches[0].group(2)

(<re.Match object; span=(35, 71), match='1. Name and territory of the Union.—'>,
 35,
 '1',
 'Name and territory of the Union.')

35 = matches[0].start(), the index of the first matched character (the 1 of 1. Name...). </br>
71 = matches[0].end(), the index just after the last matched character (one past the —).

In [17]:
articles = []
for i, m in enumerate(matches):
    end = matches[i + 1].start() if i + 1 < len(matches) else len(articles_text)
    articles.append({
        "number": m.group(1),
        "title":  " ".join(m.group(2).split()).rstrip("."),   # join the wrapped lines
        "body":   articles_text[m.end():end].strip(),
    })

print(len(articles))

500


In [20]:
articles[3]

{'number': '3',
 'title': 'Formation of new States and alteration of areas, boundaries or names of existing States',
 'body': 'Parliament may by law—\n(a) form a new State by separation of territory from any\nState or by uniting two or more States or parts of States or by\nuniting any territory to a part of any State;\n(b) increase the area of any State;\n(c) diminish the area of any State;\n(d) alter the boundaries of any State;\n(e) alter the name of any State:\n[Provided that no Bill for the purpose shall be introduced in\neither House of Parliament except on the recommendation of the\nPresident and unless, where the proposal contained in the Bill affects the\narea, boundaries or name of any of the States ***, the Bill has been\nreferred by the President to the Legislature of that State for expressing\nits views thereon within such period as may be specified in the reference\nor within such further period as the President may allow and the period\nso specified or allowed has expired